In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

file_path = "/content/drive/MyDrive/capstone-project-1/data/raw/PS_20174392719_1491204439457_log.csv"

if os.path.exists(file_path):
    print("Dataset found successfully!")
    print("File size:", round(os.path.getsize(file_path) / (1024 * 1024), 2), "MB")
else:
    print("Dataset not found. Please check the Google Drive folder.")

In [ ]:
import pandas as pd

df = pd.read_csv(file_path, nrows=5)

print("Dataset loaded successfully!")

In [ ]:
df.head()

In [ ]:
print("Total columns:", len(df.columns))

print("\nColumn names:")

for column in df.columns:
    print(column)

In [ ]:
print("Column data types:\n")
print(df.dtypes)

In [ ]:
print("Missing values in each column:\n")
print(df.isnull().sum())

In [ ]:
print("Dataset shape:", df.shape)

print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

In [ ]:
total_rows = 0

for chunk in pd.read_csv(file_path, chunksize=100000):
    total_rows += len(chunk)

print("Total transactions:", total_rows)

In [ ]:
missing_values = None

for chunk in pd.read_csv(file_path, chunksize=100000):

    chunk_missing = chunk.isnull().sum()

    if missing_values is None:
        missing_values = chunk_missing
    else:
        missing_values += chunk_missing

print("Missing values in entire dataset:")
print(missing_values)

In [ ]:
import sqlite3

conn = sqlite3.connect("/content/paysim_duplicates.db")

columns = list(df.columns)

conn.execute("""
CREATE TABLE IF NOT EXISTS seen_rows (
    row_hash TEXT PRIMARY KEY
)
""")

duplicate_count = 0

for chunk in pd.read_csv(file_path, chunksize=100000):

    row_hashes = pd.util.hash_pandas_object(
        chunk,
        index=False
    ).astype(str)

    for row_hash in row_hashes:
        cursor = conn.execute(
            "INSERT OR IGNORE INTO seen_rows (row_hash) VALUES (?)",
            (row_hash,)
        )

        if cursor.rowcount == 0:
            duplicate_count += 1

    conn.commit()

print("Potential duplicate rows:", duplicate_count)

conn.close()

In [ ]:
fraud_counts = {0: 0, 1: 0}

for chunk in pd.read_csv(
    file_path,
    usecols=["isFraud"],
    chunksize=100000
):
    counts = chunk["isFraud"].value_counts()

    for label, count in counts.items():
        fraud_counts[label] = fraud_counts.get(label, 0) + count

print("Genuine transactions:", fraud_counts[0])
print("Fraud transactions:", fraud_counts[1])

fraud_percentage = fraud_counts[1] / sum(fraud_counts.values()) * 100

print("Fraud percentage:", round(fraud_percentage, 4), "%")

In [ ]:
df.describe()

In [ ]:
amount_data = pd.read_csv(
    file_path,
    usecols=["amount"]
)

print(amount_data["amount"].describe())

In [ ]:
pd.set_option("display.float_format", "{:,.2f}".format)

print(amount_data["amount"].describe())

In [ ]:
balance_columns = [
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

balance_data = pd.read_csv(
    file_path,
    usecols=balance_columns
)

print(balance_data.describe())

In [ ]:
step_data = pd.read_csv(
    file_path,
    usecols=["step"]
)

print(step_data["step"].describe())

In [ ]:
import pandas as pd

amounts = pd.Series([100, 200, 300, 400, 500, 600, 700, 10000])

Q1 = amounts.quantile(0.25)
Q3 = amounts.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

print("Outliers:")
print(amounts[(amounts < lower_bound) | (amounts > upper_bound)])

In [ ]:
Q1 = amount_data["amount"].quantile(0.25)
Q3 = amount_data["amount"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = amount_data[
    (amount_data["amount"] < lower_bound) |
    (amount_data["amount"] > upper_bound)
]

print("Lower Bound:", round(lower_bound, 2))
print("Upper Bound:", round(upper_bound, 2))
print("Total Outliers:", len(outliers))
print("Outlier Percentage:", round(len(outliers) / len(amount_data) * 100, 2), "%")

In [ ]:
outlier_fraud_counts = {0: 0, 1: 0}

for chunk in pd.read_csv(
    file_path,
    usecols=["amount", "isFraud"],
    chunksize=100000
):
    is_outlier = (
        (chunk["amount"] < lower_bound) |
        (chunk["amount"] > upper_bound)
    )

    outlier_rows = chunk[is_outlier]

    counts = outlier_rows["isFraud"].value_counts()

    for label, count in counts.items():
        outlier_fraud_counts[label] += count

print("Genuine outlier transactions:", outlier_fraud_counts[0])
print("Fraud outlier transactions:", outlier_fraud_counts[1])

In [ ]:
import matplotlib.pyplot as plt

labels = ["Genuine", "Fraud"]
counts = [fraud_counts[0], fraud_counts[1]]

plt.figure(figsize=(7, 4))

plt.bar(labels, counts, color=["green", "red"])

plt.title("Genuine vs Fraud Transactions")
plt.xlabel("Transaction Category")
plt.ylabel("Number of Transactions")

plt.show()

In [ ]:
type_counts = pd.DataFrame()

for chunk in pd.read_csv(
    file_path,
    usecols=["type", "isFraud"],
    chunksize=100000
):
    summary = chunk.groupby("type")["isFraud"].agg(
        total="count",
        fraud="sum"
    )

    type_counts = type_counts.add(
        summary,
        fill_value=0
    )

type_counts = type_counts.fillna(0).astype(int)

type_counts["fraud_rate"] = (
    type_counts["fraud"] / type_counts["total"] * 100
)

print(type_counts)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.bar(
    type_counts.index,
    type_counts["fraud"],
    color="orange"
)

plt.title("Fraud Transactions by Transaction Type")
plt.xlabel("Transaction Type")
plt.ylabel("Number of Fraud Transactions")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.bar(
    type_counts.index,
    type_counts["fraud_rate"],
    color="steelblue"
)

plt.title("Fraud Rate by Transaction Type")
plt.xlabel("Transaction Type")
plt.ylabel("Fraud Rate (%)")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(9, 5))

plt.hist(
    np.log1p(amount_data["amount"]),
    bins=50,
    color="purple",
    edgecolor="black"
)

plt.title("Distribution of Transaction Amounts (Log Scale)")
plt.xlabel("Log(1 + Transaction Amount)")
plt.ylabel("Number of Transactions")

plt.show()

In [ ]:
fraud_by_step = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    file_path,
    usecols=["step", "isFraud"],
    chunksize=100000
):
    step_counts = chunk.groupby("step")["isFraud"].sum()

    fraud_by_step = fraud_by_step.add(
        step_counts,
        fill_value=0
    )

fraud_by_step = fraud_by_step.fillna(0).astype(int).sort_index()

print(fraud_by_step.head(10))
print("Total fraud transactions:", fraud_by_step.sum())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    fraud_by_step.index,
    fraud_by_step.values,
    color="red",
    linewidth=1.5
)

plt.title("Fraud Transactions Over Simulation Time")
plt.xlabel("Simulation Step (Hour)")
plt.ylabel("Number of Fraud Transactions")

plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
import pandas as pd

numeric_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

correlation_data = pd.read_csv(
    file_path,
    usecols=numeric_columns
)

correlation_matrix = correlation_data.corr()

print(correlation_matrix)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Heatmap of Transaction Features")

plt.show()

In [ ]:
zero_balance_count = (balance_data["newbalanceOrig"] == 0).sum()

total_transactions = len(balance_data)

zero_balance_percentage = (
    zero_balance_count / total_transactions
) * 100

print("Transactions with zero sender balance:", zero_balance_count)
print("Percentage:", round(zero_balance_percentage, 2), "%")

In [ ]:
zero_balance_fraud = {0: 0, 1: 0}

for chunk in pd.read_csv(
    file_path,
    usecols=["newbalanceOrig", "isFraud"],
    chunksize=100000
):
    zero_rows = chunk[chunk["newbalanceOrig"] == 0]

    counts = zero_rows["isFraud"].value_counts()

    for label, count in counts.items():
        zero_balance_fraud[label] += count

print("Genuine transactions with zero ending balance:",
      zero_balance_fraud[0])

print("Fraud transactions with zero ending balance:",
      zero_balance_fraud[1])

In [ ]:
balance_check = pd.read_csv(
    file_path,
    usecols=[
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig"
    ]
)

balance_check["expected_balance"] = (
    balance_check["oldbalanceOrg"] -
    balance_check["amount"]
)

balance_check["balance_difference"] = (
    balance_check["newbalanceOrig"] -
    balance_check["expected_balance"]
)

print(balance_check.head(10))

In [ ]:
transaction_8 = pd.read_csv(
    file_path,
    nrows=9
).iloc[8]

print(transaction_8)

In [ ]:
mismatch_counts = {}

for chunk in pd.read_csv(
    file_path,
    usecols=["type", "amount", "oldbalanceOrg", "newbalanceOrig"],
    chunksize=100000
):
    expected = chunk["oldbalanceOrg"] - chunk["amount"]

    mismatch = (
        chunk["newbalanceOrig"] - expected
    ).abs() > 0.01

    chunk_counts = chunk.groupby("type").size()
    chunk_mismatches = chunk[mismatch].groupby("type").size()

    for transaction_type, total in chunk_counts.items():
        if transaction_type not in mismatch_counts:
            mismatch_counts[transaction_type] = {
                "total": 0,
                "mismatches": 0
            }

        mismatch_counts[transaction_type]["total"] += total

        mismatch_counts[transaction_type]["mismatches"] += (
            chunk_mismatches.get(transaction_type, 0)
        )

for transaction_type, values in mismatch_counts.items():
    percentage = values["mismatches"] / values["total"] * 100

    print(
        transaction_type,
        "| Total:", values["total"],
        "| Mismatches:", values["mismatches"],
        "| Percentage:", round(percentage, 2), "%"
    )

In [ ]:
cash_in_total = 0
cash_in_mismatches = 0

for chunk in pd.read_csv(
    file_path,
    usecols=["type", "amount", "oldbalanceOrg", "newbalanceOrig"],
    chunksize=100000
):
    cash_in = chunk[chunk["type"] == "CASH_IN"]

    expected = (
        cash_in["oldbalanceOrg"] + cash_in["amount"]
    )

    mismatch = (
        cash_in["newbalanceOrig"] - expected
    ).abs() > 0.01

    cash_in_total += len(cash_in)
    cash_in_mismatches += mismatch.sum()

print("Total CASH_IN transactions:", cash_in_total)
print("CASH_IN mismatches:", cash_in_mismatches)
print(
    "Mismatch percentage:",
    round(cash_in_mismatches / cash_in_total * 100, 2),
    "%"
)

In [ ]:
for chunk in pd.read_csv(
    file_path,
    usecols=[
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "isFraud"
    ],
    chunksize=100000
):
    cash_in = chunk[chunk["type"] == "CASH_IN"].copy()

    cash_in["expected_balance"] = (
        cash_in["oldbalanceOrg"] + cash_in["amount"]
    )

    cash_in["difference"] = (
        cash_in["newbalanceOrig"] -
        cash_in["expected_balance"]
    )

    mismatches = cash_in[
        cash_in["difference"].abs() > 0.01
    ]

    if len(mismatches) > 0:
        print(mismatches.head(5).to_string(index=False))
        break

In [ ]:
cash_in_total = 0
meaningful_mismatches = 0

for chunk in pd.read_csv(
    file_path,
    usecols=["type", "amount", "oldbalanceOrg", "newbalanceOrig"],
    chunksize=100000
):
    cash_in = chunk[chunk["type"] == "CASH_IN"]

    expected = (
        cash_in["oldbalanceOrg"] + cash_in["amount"]
    )

    difference = (
        cash_in["newbalanceOrig"] - expected
    ).abs()

    meaningful_mismatches += (difference > 0.02).sum()
    cash_in_total += len(cash_in)

print("Total CASH_IN transactions:", cash_in_total)
print("Mismatches greater than 0.02:", meaningful_mismatches)
print(
    "Mismatch percentage:",
    round(meaningful_mismatches / cash_in_total * 100, 2),
    "%"
)

In [ ]:
for chunk in pd.read_csv(
    file_path,
    usecols=[
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "isFraud"
    ],
    chunksize=100000
):
    cash_in = chunk[chunk["type"] == "CASH_IN"].copy()

    cash_in["expected_balance"] = (
        cash_in["oldbalanceOrg"] + cash_in["amount"]
    )

    cash_in["difference"] = (
        cash_in["newbalanceOrig"] -
        cash_in["expected_balance"]
    )

    mismatches = cash_in[
        cash_in["difference"].abs() > 0.02
    ]

    if not mismatches.empty:
        print(mismatches.head(10).to_string(index=False))
        break

In [ ]:
total_mismatches = 0
unchanged_balance_count = 0

for chunk in pd.read_csv(
    file_path,
    usecols=[
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig"
    ],
    chunksize=100000
):
    cash_in = chunk[chunk["type"] == "CASH_IN"]

    expected = cash_in["oldbalanceOrg"] + cash_in["amount"]

    mismatches = cash_in[
        (cash_in["newbalanceOrig"] - expected).abs() > 0.02
    ]

    total_mismatches += len(mismatches)

    unchanged_balance_count += (
        (
            mismatches["newbalanceOrig"] -
            mismatches["oldbalanceOrg"]
        ).abs() <= 0.02
    ).sum()

print("Total CASH_IN mismatches:", total_mismatches)
print("Mismatches with unchanged balance:", unchanged_balance_count)
print(
    "Other mismatch patterns:",
    total_mismatches - unchanged_balance_count
)

In [ ]:
cash_out_total = 0
cash_out_mismatches = 0

for chunk in pd.read_csv(
    file_path,
    usecols=[
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig"
    ],
    chunksize=100000
):
    cash_out = chunk[chunk["type"] == "CASH_OUT"]

    expected = (
        cash_out["oldbalanceOrg"] - cash_out["amount"]
    )

    difference = (
        cash_out["newbalanceOrig"] - expected
    ).abs()

    cash_out_total += len(cash_out)
    cash_out_mismatches += (difference > 0.02).sum()

print("Total CASH_OUT transactions:", cash_out_total)
print("CASH_OUT mismatches:", cash_out_mismatches)
print(
    "Mismatch percentage:",
    round(cash_out_mismatches / cash_out_total * 100, 2),
    "%"
)

In [ ]:
for chunk in pd.read_csv(
    file_path,
    usecols=[
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "isFraud"
    ],
    chunksize=100000
):
    cash_out = chunk[chunk["type"] == "CASH_OUT"].copy()

    cash_out["expected_balance"] = (
        cash_out["oldbalanceOrg"] - cash_out["amount"]
    )

    cash_out["difference"] = (
        cash_out["newbalanceOrig"] -
        cash_out["expected_balance"]
    )

    mismatches = cash_out[
        cash_out["difference"].abs() > 0.02
    ]

    if not mismatches.empty:
        print(mismatches.head(5).to_string(index=False))
        break

In [ ]:
total_mismatches = 0
zero_balance_pattern = 0

for chunk in pd.read_csv(
    file_path,
    usecols=[
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig"
    ],
    chunksize=100000
):
    cash_out = chunk[chunk["type"] == "CASH_OUT"]

    expected = cash_out["oldbalanceOrg"] - cash_out["amount"]

    mismatch = (
        cash_out["newbalanceOrig"] - expected
    ).abs() > 0.02

    pattern = (
        mismatch
        & (cash_out["amount"] > cash_out["oldbalanceOrg"])
        & (cash_out["newbalanceOrig"] == 0)
    )

    total_mismatches += mismatch.sum()
    zero_balance_pattern += pattern.sum()

print("Total CASH_OUT mismatches:", total_mismatches)
print("Mismatches following zero-balance pattern:", zero_balance_pattern)
print("Other mismatch patterns:", total_mismatches - zero_balance_pattern)

In [ ]:
for chunk in pd.read_csv(
    file_path,
    usecols=[
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "isFraud"
    ],
    chunksize=100000
):
    cash_out = chunk[chunk["type"] == "CASH_OUT"].copy()

    expected = cash_out["oldbalanceOrg"] - cash_out["amount"]

    mismatch = (
        cash_out["newbalanceOrig"] - expected
    ).abs() > 0.02

    zero_balance_pattern = (
        (cash_out["amount"] > cash_out["oldbalanceOrg"])
        & (cash_out["newbalanceOrig"] == 0)
    )

    other_mismatches = cash_out[
        mismatch & ~zero_balance_pattern
    ].copy()

    if not other_mismatches.empty:
        other_mismatches["expected_balance"] = (
            other_mismatches["oldbalanceOrg"]
            - other_mismatches["amount"]
        )

        other_mismatches["difference"] = (
            other_mismatches["newbalanceOrig"]
            - other_mismatches["expected_balance"]
        )

        print(other_mismatches.to_string(index=False))

In [ ]:
pre_transaction_features = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "nameDest",
    "oldbalanceDest"
]

excluded_columns = [
    "newbalanceOrig",
    "newbalanceDest",
    "isFlaggedFraud",
    "isFraud"
]

print("Candidate pre-transaction features:")
print(pre_transaction_features)

print("\nExcluded columns:")
print(excluded_columns)

In [ ]:
account_data = pd.read_csv(
    file_path,
    usecols=["nameOrig", "nameDest"]
)

print("Total transactions:", len(account_data))

print(
    "Unique sender accounts:",
    account_data["nameOrig"].nunique()
)

print(
    "Unique receiver accounts:",
    account_data["nameDest"].nunique()
)

In [ ]:
baseline_features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest"
]

target_column = "isFraud"

print("Baseline ML features:")
for feature in baseline_features:
    print("-", feature)

print("\nTarget column:", target_column)

print("\nTotal baseline features:", len(baseline_features))

In [ ]:
step_info = pd.read_csv(
    file_path,
    usecols=["step"]
)

print("Minimum step:", step_info["step"].min())
print("Maximum step:", step_info["step"].max())
print("Total unique steps:", step_info["step"].nunique())

In [ ]:
split_counts = {
    "Train": {"total": 0, "fraud": 0},
    "Validation": {"total": 0, "fraud": 0},
    "Test": {"total": 0, "fraud": 0}
}

for chunk in pd.read_csv(
    file_path,
    usecols=["step", "isFraud"],
    chunksize=100000
):
    groups = {
        "Train": chunk[chunk["step"] <= 520],
        "Validation": chunk[
            (chunk["step"] >= 521) &
            (chunk["step"] <= 631)
        ],
        "Test": chunk[chunk["step"] >= 632]
    }

    for name, data in groups.items():
        split_counts[name]["total"] += len(data)
        split_counts[name]["fraud"] += data["isFraud"].sum()

for name, values in split_counts.items():
    print(
        name,
        "| Total:", values["total"],
        "| Fraud:", values["fraud"]
    )

In [ ]:
time_ranges = [
    (1, 100),
    (101, 200),
    (201, 300),
    (301, 400),
    (401, 500),
    (501, 600),
    (601, 743)
]

for start, end in time_ranges:
    count = (
        (step_info["step"] >= start) &
        (step_info["step"] <= end)
    ).sum()

    print(
        f"Steps {start}-{end}: {count:,} transactions"
    )

In [ ]:
transactions_per_step = (
    step_info["step"]
    .value_counts()
    .sort_index()
)

cumulative_transactions = transactions_per_step.cumsum()

total_transactions = len(step_info)

train_target = total_transactions * 0.70
validation_target = total_transactions * 0.85

train_end_step = cumulative_transactions[
    cumulative_transactions >= train_target
].index[0]

validation_end_step = cumulative_transactions[
    cumulative_transactions >= validation_target
].index[0]

print("Training ends at step:", train_end_step)
print("Validation ends at step:", validation_end_step)
print("Testing starts at step:", validation_end_step + 1)

In [ ]:
split_counts = {
    "Train": {"total": 0, "fraud": 0},
    "Validation": {"total": 0, "fraud": 0},
    "Test": {"total": 0, "fraud": 0}
}

for chunk in pd.read_csv(
    file_path,
    usecols=["step", "isFraud"],
    chunksize=100000
):
    groups = {
        "Train": chunk[chunk["step"] <= 323],
        "Validation": chunk[
            (chunk["step"] >= 324) &
            (chunk["step"] <= 378)
        ],
        "Test": chunk[chunk["step"] >= 379]
    }

    for name, data in groups.items():
        split_counts[name]["total"] += len(data)
        split_counts[name]["fraud"] += data["isFraud"].sum()

for name, values in split_counts.items():
    fraud_rate = values["fraud"] / values["total"] * 100

    print(
        name,
        "| Total:", values["total"],
        "| Fraud:", values["fraud"],
        "| Fraud rate:", round(fraud_rate, 4), "%"
    )

In [ ]:
import pandas as pd

sample = pd.DataFrame({
    "type": [
        "PAYMENT",
        "TRANSFER",
        "CASH_OUT",
        "PAYMENT",
        "CASH_IN"
    ]
})

encoded = pd.get_dummies(
    sample,
    columns=["type"],
    dtype=int
)

print(encoded)

In [ ]:
model_columns = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "isFraud"
]

model_data = pd.read_csv(
    file_path,
    usecols=model_columns
)

print("Dataset shape:", model_data.shape)
print("\nFirst 5 rows:")
print(model_data.head())

In [ ]:
train_data = model_data[
    model_data["step"] <= 323
].copy()

val_data = model_data[
    (model_data["step"] >= 324) &
    (model_data["step"] <= 378)
].copy()

test_data = model_data[
    model_data["step"] >= 379
].copy()

print("Training shape:", train_data.shape)
print("Validation shape:", val_data.shape)
print("Testing shape:", test_data.shape)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/capstone-project-1/data/raw/PS_20174392719_1491204439457_log.csv"

model_columns = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "isFraud"
]

model_data = pd.read_csv(
    file_path,
    usecols=model_columns
)

train_data = model_data[
    model_data["step"] <= 323
].copy()

val_data = model_data[
    (model_data["step"] >= 324) &
    (model_data["step"] <= 378)
].copy()

test_data = model_data[
    model_data["step"] >= 379
].copy()

print("Training shape:", train_data.shape)
print("Validation shape:", val_data.shape)
print("Testing shape:", test_data.shape)

In [ ]:
X_train = train_data.drop(columns=["isFraud"])
y_train = train_data["isFraud"]

X_val = val_data.drop(columns=["isFraud"])
y_val = val_data["isFraud"]

X_test = test_data.drop(columns=["isFraud"])
y_test = test_data["isFraud"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder.fit(X_train[["type"]])

print("Learned transaction categories:")
print(encoder.categories_)

print("\nEncoded feature names:")
print(encoder.get_feature_names_out(["type"]))

In [ ]:
X_train_type = encoder.transform(X_train[["type"]])

X_val_type = encoder.transform(X_val[["type"]])

X_test_type = encoder.transform(X_test[["type"]])

print("Training encoded shape:", X_train_type.shape)
print("Validation encoded shape:", X_val_type.shape)
print("Testing encoded shape:", X_test_type.shape)

In [ ]:
from scipy.sparse import csr_matrix, hstack
import numpy as np

numeric_columns = [
    "step",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest"
]

X_train_numeric = csr_matrix(
    X_train[numeric_columns].to_numpy(dtype=np.float32)
)

X_train_final = hstack(
    [X_train_numeric, X_train_type],
    format="csr",
    dtype=np.float32
)

print("Final training shape:", X_train_final.shape)
print("Data type:", X_train_final.dtype)

In [ ]:
X_val_numeric = csr_matrix(
    X_val[numeric_columns].to_numpy(dtype=np.float32)
)

X_test_numeric = csr_matrix(
    X_test[numeric_columns].to_numpy(dtype=np.float32)
)

X_val_final = hstack(
    [X_val_numeric, X_val_type],
    format="csr",
    dtype=np.float32
)

X_test_final = hstack(
    [X_test_numeric, X_test_type],
    format="csr",
    dtype=np.float32
)

print("Training shape:", X_train_final.shape)
print("Validation shape:", X_val_final.shape)
print("Testing shape:", X_test_final.shape)

In [ ]:
import numpy as np

for name, X in [
    ("Training", X_train_final),
    ("Validation", X_val_final),
    ("Testing", X_test_final)
]:
    invalid_count = np.count_nonzero(
        ~np.isfinite(X.data)
    )

    print(f"{name} invalid values: {invalid_count}")

In [ ]:
from sklearn.linear_model import LogisticRegression
import time

# Exclude oldbalanceDest (column index 3)
selected_columns = [0, 1, 2, 4, 5, 6, 7, 8]

X_train_baseline = X_train_final[:, selected_columns]
X_val_baseline = X_val_final[:, selected_columns]

print("Training shape:", X_train_baseline.shape)
print("Validation shape:", X_val_baseline.shape)

model = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=100,
    random_state=42
)

print("\nModel initialized successfully!")

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

# First three columns are numerical:
# step, amount, oldbalanceOrg

scaler = StandardScaler()

# Learn scaling parameters from training data only
X_train_scaled_numeric = scaler.fit_transform(
    X_train_baseline[:, :3].toarray()
)

# Apply the same scaling to validation data
X_val_scaled_numeric = scaler.transform(
    X_val_baseline[:, :3].toarray()
)

# Combine scaled numerical features with encoded type
X_train_scaled = hstack(
    [X_train_scaled_numeric, X_train_baseline[:, 3:]],
    format="csr"
)

X_val_scaled = hstack(
    [X_val_scaled_numeric, X_val_baseline[:, 3:]],
    format="csr"
)

print("Training shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)

print("\nFeature scaling completed successfully!")

In [ ]:
import time
from sklearn.exceptions import ConvergenceWarning
import warnings

start_time = time.time()

print("Training Logistic Regression model...")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)

    model.fit(X_train_scaled, y_train)

training_time = time.time() - start_time

print("\nModel training finished!")
print(f"Training time: {training_time:.2f} seconds")

convergence_warnings = [
    w for w in caught
    if issubclass(w.category, ConvergenceWarning)
]

print("Convergence warnings:", len(convergence_warnings))

In [ ]:
for warning in convergence_warnings:
    print("Warning:", warning.message)

print("\nIterations used:", model.n_iter_)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# Predict fraud probabilities on validation data
y_val_prob = model.predict_proba(X_val_scaled)[:, 1]

# Initial classification threshold: 0.5
y_val_pred = (y_val_prob >= 0.5).astype(int)

print("Precision:", round(precision_score(y_val, y_val_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_val, y_val_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_val, y_val_pred, zero_division=0), 4))
print("PR-AUC (Average Precision):", round(average_precision_score(y_val, y_val_prob), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]

print("Threshold | Precision | Recall | F1 | Alerts")
print("-" * 58)

for threshold in thresholds:
    predictions = (y_val_prob >= threshold).astype(int)

    precision = precision_score(
        y_val, predictions, zero_division=0
    )
    recall = recall_score(
        y_val, predictions, zero_division=0
    )
    f1 = f1_score(
        y_val, predictions, zero_division=0
    )

    alerts = predictions.sum()

    print(
        f"{threshold:9.2f} | "
        f"{precision:9.4f} | "
        f"{recall:6.4f} | "
        f"{f1:6.4f} | "
        f"{alerts}"
    )

In [ ]:
import pandas as pd

baseline_results = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Threshold": [0.90],
    "Precision": [0.0938],
    "Recall": [0.2323],
    "F1_Score": [0.1336],
    "PR_AUC": [0.0792],
    "Converged": [False]
})

baseline_results.to_csv(
    "/content/logistic_regression_baseline.csv",
    index=False
)

print(baseline_results)
print("\nBaseline results saved successfully!")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
import warnings
import time

improved_model = LogisticRegression(
    class_weight="balanced",
    solver="lbfgs",
    max_iter=300,
    random_state=42
)

start_time = time.time()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)

    improved_model.fit(X_train_scaled, y_train)

training_time = time.time() - start_time

convergence_warnings_new = [
    w for w in caught
    if issubclass(w.category, ConvergenceWarning)
]

print("Training completed!")
print(f"Training time: {training_time:.2f} seconds")
print("Iterations used:", improved_model.n_iter_)
print("Convergence warnings:", len(convergence_warnings_new))

for warning in convergence_warnings_new:
    print("Warning:", warning.message)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# Predict fraud probabilities
y_val_prob_improved = improved_model.predict_proba(
    X_val_scaled
)[:, 1]

# Use the same 0.90 threshold for comparison
y_val_pred_improved = (
    y_val_prob_improved >= 0.90
).astype(int)

print("Precision:", round(
    precision_score(y_val, y_val_pred_improved, zero_division=0), 4
))

print("Recall:", round(
    recall_score(y_val, y_val_pred_improved, zero_division=0), 4
))

print("F1-score:", round(
    f1_score(y_val, y_val_pred_improved, zero_division=0), 4
))

print("PR-AUC:", round(
    average_precision_score(y_val, y_val_prob_improved), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_improved))


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.50, 0.70, 0.80, 0.90, 0.95, 0.99]

print("Threshold | Precision | Recall | F1 | Alerts")
print("-" * 58)

for threshold in thresholds:
    predictions = (
        y_val_prob_improved >= threshold
    ).astype(int)

    precision = precision_score(
        y_val, predictions, zero_division=0
    )

    recall = recall_score(
        y_val, predictions, zero_division=0
    )

    f1 = f1_score(
        y_val, predictions, zero_division=0
    )

    alerts = predictions.sum()

    print(
        f"{threshold:9.2f} | "
        f"{precision:9.4f} | "
        f"{recall:6.4f} | "
        f"{f1:6.4f} | "
        f"{alerts}"
    )

In [ ]:
improved_results = pd.DataFrame({
    "Model": ["Logistic Regression (lbfgs)"],
    "Threshold": [0.99],
    "Precision": [0.0477],
    "Recall": [0.3121],
    "F1_Score": [0.0827],
    "PR_AUC": [0.1168],
    "Converged": [True]
})

model_comparison = pd.concat(
    [baseline_results, improved_results],
    ignore_index=True
)

model_comparison.to_csv(
    "/content/model_comparison.csv",
    index=False
)

print(model_comparison.to_string(index=False))
print("\nModel comparison saved successfully!")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=12,
    min_samples_leaf=20,
    class_weight="balanced_subsample",
    n_jobs=2,
    random_state=42
)

print("Random Forest initialized successfully!")
print("Number of trees:", rf_model.n_estimators)
print("Maximum tree depth:", rf_model.max_depth)
print("Minimum samples per leaf:", rf_model.min_samples_leaf)

print("\nTraining input shape:", X_train_baseline.shape)

In [ ]:
import time

start_time = time.time()

print("Training Random Forest model...")

rf_model.fit(X_train_baseline, y_train)

training_time_rf = time.time() - start_time

print("\nRandom Forest training completed!")
print(f"Training time: {training_time_rf:.2f} seconds")
print("Number of trained trees:", len(rf_model.estimators_))

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# Predict fraud probabilities
y_val_prob_rf = rf_model.predict_proba(
    X_val_baseline
)[:, 1]

# Initial threshold
y_val_pred_rf = (
    y_val_prob_rf >= 0.5
).astype(int)

print("Precision:", round(
    precision_score(y_val, y_val_pred_rf, zero_division=0), 4
))

print("Recall:", round(
    recall_score(y_val, y_val_pred_rf, zero_division=0), 4
))

print("F1-score:", round(
    f1_score(y_val, y_val_pred_rf, zero_division=0), 4
))

print("PR-AUC:", round(
    average_precision_score(y_val, y_val_prob_rf), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_rf))

In [ ]:
import joblib
import os

model_dir = "/content/drive/MyDrive/capstone-project-1/artifacts/models"

os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(
    model_dir,
    "random_forest_v1.joblib"
)

joblib.dump(rf_model, model_path)

print("Random Forest model saved successfully!")
print("Location:", model_path)
print("File size (MB):", round(
    os.path.getsize(model_path) / (1024 * 1024), 2
))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

print("Threshold | Precision | Recall | F1 | Alerts")
print("-" * 58)

for threshold in thresholds:
    predictions = (
        y_val_prob_rf >= threshold
    ).astype(int)

    precision = precision_score(
        y_val, predictions, zero_division=0
    )

    recall = recall_score(
        y_val, predictions, zero_division=0
    )

    f1 = f1_score(
        y_val, predictions, zero_division=0
    )

    alerts = predictions.sum()

    print(
        f"{threshold:9.2f} | "
        f"{precision:9.4f} | "
        f"{recall:6.4f} | "
        f"{f1:6.4f} | "
        f"{alerts}"
    )

In [ ]:
rf_results = pd.DataFrame({
    "Model": ["Random Forest"],
    "Threshold": [0.90],
    "Precision": [0.8271],
    "Recall": [0.4326],
    "F1_Score": [0.5681],
    "PR_AUC": [0.6255],
    "Converged": [True]
})

model_comparison = pd.concat(
    [model_comparison, rf_results],
    ignore_index=True
)

# Save comparison to Google Drive
comparison_path = (
    "/content/drive/MyDrive/capstone-project-1/"
    "artifacts/model_comparison.csv"
)

model_comparison.to_csv(comparison_path, index=False)

print(model_comparison.to_string(index=False))
print("\nComparison saved to:", comparison_path)

In [ ]:
import pandas as pd

feature_names = [
    "step",
    "amount",
    "oldbalanceOrg",
    "type_CASH_IN",
    "type_CASH_OUT",
    "type_DEBIT",
    "type_PAYMENT",
    "type_TRANSFER"
]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.to_string(index=False))

In [ ]:
for name, labels in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Testing", y_test)
]:
    fraud_count = int(labels.sum())
    fraud_rate = labels.mean() * 100

    print(
        f"{name}: {fraud_count:,} fraud cases, "
        f"fraud rate = {fraud_rate:.4f}%"
    )

In [ ]:
variables_to_check = [
    "X_test",
    "X_test_encoded",
    "encoder",
    "ohe",
    "X_train_baseline",
    "X_val_baseline",
    "rf_model"
]

for variable in variables_to_check:
    print(
        variable,
        ":", "Available" if variable in globals()
        else "Not available"
    )

In [ ]:
from scipy.sparse import csr_matrix, hstack
import numpy as np

# Same numerical features used during training
numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest"
]

# Transform transaction types using existing encoder
X_test_type_encoded = encoder.transform(
    X_test[["type"]]
)

# Convert numerical features to sparse matrix
X_test_numeric = csr_matrix(
    X_test[numeric_features].to_numpy(
        dtype=np.float32
    )
)

# Combine numerical and encoded features
X_test_encoded = hstack(
    [X_test_numeric, X_test_type_encoded],
    format="csr"
)

# Remove oldbalanceDest, same as training
baseline_feature_indices = [0, 1, 2, 4, 5, 6, 7, 8]

X_test_baseline = X_test_encoded[
    :, baseline_feature_indices
].tocsr()

print("Training shape:", X_train_baseline.shape)
print("Validation shape:", X_val_baseline.shape)
print("Testing shape:", X_test_baseline.shape)

print(
    "\nFeature counts match:",
    X_train_baseline.shape[1]
    == X_val_baseline.shape[1]
    == X_test_baseline.shape[1]
)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# Predict fraud probabilities on unseen test data
y_test_prob_rf = rf_model.predict_proba(
    X_test_baseline
)[:, 1]

# Apply our selected validation threshold
rf_threshold = 0.90

y_test_pred_rf = (
    y_test_prob_rf >= rf_threshold
).astype(int)

# Evaluate model
print("Random Forest - Final Test Results")
print("-" * 45)

print("Threshold:", rf_threshold)

print("Precision:", round(
    precision_score(y_test, y_test_pred_rf, zero_division=0), 4
))

print("Recall:", round(
    recall_score(y_test, y_test_pred_rf, zero_division=0), 4
))

print("F1-score:", round(
    f1_score(y_test, y_test_pred_rf, zero_division=0), 4
))

print("PR-AUC:", round(
    average_precision_score(y_test, y_test_prob_rf), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_rf))

print("\nTotal fraud alerts:", int(y_test_pred_rf.sum()))

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print(
    "Device:",
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Select a smaller, stratified training sample
all_indices = np.arange(X_train_scaled.shape[0])

sample_indices, _ = train_test_split(
    all_indices,
    train_size=200_000,
    stratify=np.asarray(y_train),
    random_state=42
)

# Prepare sampled training features and labels
X_dl_train = X_train_scaled[sample_indices].toarray().astype(np.float32)
y_dl_train = np.asarray(y_train)[sample_indices].astype(np.float32)

print("DL training shape:", X_dl_train.shape)
print("Fraud cases:", int(y_dl_train.sum()))
print("Genuine cases:", int(len(y_dl_train) - y_dl_train.sum()))
print("Fraud percentage:", round(y_dl_train.mean() * 100, 4))

In [ ]:
import torch
import torch.nn as nn

class FraudNeuralNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)


# Create neural network
dl_model = FraudNeuralNetwork()

# Handle fraud class imbalance
fraud_count = y_dl_train.sum()
genuine_count = len(y_dl_train) - fraud_count

positive_weight = torch.tensor(
    [genuine_count / fraud_count],
    dtype=torch.float32
)

# Loss function
criterion = nn.BCEWithLogitsLoss(
    pos_weight=positive_weight
)

# Optimizer
optimizer = torch.optim.Adam(
    dl_model.parameters(),
    lr=0.001
)

print(dl_model)
print("\nFraud class weight:", round(
    positive_weight.item(), 2
))

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import time

# Convert training data into PyTorch tensors
X_tensor = torch.tensor(
    X_dl_train, dtype=torch.float32
)

y_tensor = torch.tensor(
    y_dl_train, dtype=torch.float32
).reshape(-1, 1)

# Create batches
train_dataset = TensorDataset(X_tensor, y_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

# Train neural network
epochs = 5

start_time = time.time()

for epoch in range(epochs):

    dl_model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = dl_model(X_batch)

        loss = criterion(predictions, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    avg_loss = total_loss / len(train_dataset)

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"- Loss: {avg_loss:.4f}"
    )

print(
    "\nTraining completed in",
    round(time.time() - start_time, 2),
    "seconds"
)

In [ ]:
import torch
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# Switch model to evaluation mode
dl_model.eval()

# Predict validation data in batches to manage memory
validation_probabilities = []

with torch.no_grad():
    for start in range(0, X_val_scaled.shape[0], 10000):

        batch = X_val_scaled[
            start:start + 10000
        ].toarray().astype(np.float32)

        batch_tensor = torch.from_numpy(batch)

        logits = dl_model(batch_tensor)

        probabilities = torch.sigmoid(logits)

        validation_probabilities.extend(
            probabilities.numpy().flatten()
        )

y_val_prob_dl = np.array(validation_probabilities)

# Initial threshold
threshold = 0.50

y_val_pred_dl = (
    y_val_prob_dl >= threshold
).astype(int)

print("Deep Learning - Validation Results")
print("-" * 42)

print("Precision:", round(
    precision_score(y_val, y_val_pred_dl, zero_division=0), 4
))

print("Recall:", round(
    recall_score(y_val, y_val_pred_dl, zero_division=0), 4
))

print("F1-score:", round(
    f1_score(y_val, y_val_pred_dl, zero_division=0), 4
))

print("PR-AUC:", round(
    average_precision_score(y_val, y_val_prob_dl), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_dl))

In [ ]:
import torch
import os

model_dir = (
    "/content/drive/MyDrive/"
    "capstone-project-1/artifacts/models"
)

os.makedirs(model_dir, exist_ok=True)

dl_model_path = os.path.join(
    model_dir,
    "fraud_neural_network_v1.pth"
)

torch.save({
    "model_state_dict": dl_model.state_dict(),
    "input_features": 8,
    "validation_threshold": 0.50,
    "validation_pr_auc": 0.0197,
    "validation_f1": 0.0096
}, dl_model_path)

print("Deep Learning model saved successfully!")
print("Location:", dl_model_path)
print("Size (KB):", round(
    os.path.getsize(dl_model_path) / 1024, 2
))

In [ ]:
import os

base_path = "/content/drive/MyDrive/capstone-project-1"

folders = [
    "data/knowledge_base",
    "artifacts/vector_db",
    "artifacts/genai_outputs",
    "src/genai"
]

for folder in folders:
    os.makedirs(
        os.path.join(base_path, folder),
        exist_ok=True
    )

print("GenAI project folders created successfully!")

for folder in folders:
    print("✓", folder)

In [ ]:
import os

knowledge_base_path = (
    "/content/drive/MyDrive/capstone-project-1/"
    "data/knowledge_base"
)

os.makedirs(knowledge_base_path, exist_ok=True)

policy_text = """
NORTHSTAR PAYMENTS
FRAUD INVESTIGATION POLICY
Document ID: NP-FRAUD-001
Version: 1.0
Synthetic demonstration document

1. PURPOSE
This policy supports human investigation of suspicious
financial transactions. A fraud score is not proof of fraud.

2. TRANSACTION AMOUNT
Investigators should review transaction amounts in context.
A large transaction alone does not establish fraud.

3. SENDER BALANCE
Investigators should compare the transaction amount with
the sender's available balance before the transaction.

4. TRANSACTION TYPE
TRANSFER and CASH_OUT transactions may require additional
review when supported by other risk indicators.

5. FRAUD RISK SCORE
Machine Learning predictions are decision-support signals.
They must not be treated as confirmed fraudulent activity.

6. HUMAN REVIEW
An authorized investigator must review the available
evidence before making consequential decisions.

7. EVIDENCE AND AUDIT
Investigation reports must distinguish verified facts,
model predictions, and unresolved questions.
All investigation activities should be recorded.

8. INSUFFICIENT EVIDENCE
If supporting evidence is missing, the AI assistant
must identify the missing information and recommend
further human investigation.
"""

file_path = os.path.join(
    knowledge_base_path,
    "fraud_investigation_policy.txt"
)

with open(file_path, "w", encoding="utf-8") as file:
    file.write(policy_text)

print("Knowledge base document created successfully!")
print("File:", file_path)
print("Characters:", len(policy_text))

In [ ]:
!pip -q install langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load our policy document
file_path = (
    "/content/drive/MyDrive/capstone-project-1/"
    "data/knowledge_base/fraud_investigation_policy.txt"
)

with open(file_path, "r", encoding="utf-8") as file:
    policy_content = file.read()

# Configure text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

# Split document into chunks
chunks = text_splitter.split_text(policy_content)

print("Document characters:", len(policy_content))
print("Total chunks created:", len(chunks))

print("\nFirst chunk:")
print(chunks[0])

print("\nSecond chunk:")
print(chunks[1] if len(chunks) > 1 else "No second chunk")

In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# Load pretrained embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Convert all 5 text chunks into numerical vectors
chunk_embeddings = embedding_model.encode(chunks)

print("Total chunks:", len(chunks))
print("Embeddings shape:", chunk_embeddings.shape)

print("\nFirst 10 values of Chunk 1 embedding:")
print(chunk_embeddings[0][:10])

In [ ]:
import chromadb

print("ChromaDB imported successfully!")
print("ChromaDB version:", chromadb.__version__)

# Create a temporary test database
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="installation_test"
)

print("ChromaDB test collection created successfully!")

In [ ]:
import chromadb

db_path = (
    "/content/drive/MyDrive/capstone-project-1/"
    "artifacts/vector_db"
)

# Create persistent ChromaDB client
chroma_client = chromadb.PersistentClient(
    path=db_path
)

# Create or reuse our collection
fraud_collection = chroma_client.get_or_create_collection(
    name="fraud_investigation_policies",
    metadata={"hnsw:space": "cosine"}
)

# Store text chunks and their embeddings
fraud_collection.upsert(
    ids=[f"policy_chunk_{i+1}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    metadatas=[
        {
            "source": "fraud_investigation_policy.txt",
            "chunk_number": i + 1
        }
        for i in range(len(chunks))
    ]
)

print("Embeddings stored successfully!")
print("Total stored chunks:", fraud_collection.count())
print("Database location:", db_path)

In [ ]:
# Investigator's question
user_question = (
    "What should I check about the sender's "
    "balance when investigating fraud?"
)

# Convert question into an embedding
question_embedding = embedding_model.encode(
    user_question
).tolist()

# Search ChromaDB
search_results = fraud_collection.query(
    query_embeddings=[question_embedding],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

print("Question:", user_question)
print("\nRetrieved policy chunks:")
print("-" * 55)

for i, document in enumerate(
    search_results["documents"][0]
):
    print(f"\nResult {i + 1}")
    print("Source:", search_results["metadatas"][0][i]["source"])
    print("Distance:", round(
        search_results["distances"][0][i], 4
    ))
    print("Content:")
    print(document)

In [ ]:
!pip -q install google-genai

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

gemini_client = genai.Client(api_key=api_key)

print("Gemini client configured successfully!")

In [ ]:
# Step 65: Generate answer using RAG context

response = test_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

print("INVESTIGATOR QUESTION:")
print(user_question)

print("\nGEMINI RAG RESPONSE:")
print(response.text)

print("\nRETRIEVED POLICY CONTEXT:")
print(context)

print("\nSOURCE:")
print("fraud_investigation_policy.txt")

In [ ]:
import os

model_path = (
    "/content/drive/MyDrive/capstone-project-1/"
    "artifacts/models/random_forest_v1.joblib"
)

if os.path.exists(model_path):
    print("Random Forest model found successfully!")
    print("Model size:", round(
        os.path.getsize(model_path) / 1024 / 1024, 2
    ), "MB")
else:
    print("Model not found. Check Google Drive path.")

In [ ]:
import joblib

# Load the previously trained ML model
fraud_model = joblib.load(model_path)

print("Random Forest model loaded successfully!")
print("Model type:", type(fraud_model).__name__)
print("Number of features:", fraud_model.n_features_in_)
print("Model classes:", fraud_model.classes_)

In [ ]:
import pandas as pd

# Example synthetic financial transaction
sample_transaction = {
    "step": 100,
    "amount": 90000.0,
    "oldbalanceOrg": 95000.0,
    "type_CASH_IN": 0,
    "type_CASH_OUT": 0,
    "type_DEBIT": 0,
    "type_PAYMENT": 0,
    "type_TRANSFER": 1
}

# Convert to DataFrame
transaction_df = pd.DataFrame([sample_transaction])

print("Sample Transaction:")
print(transaction_df.to_string(index=False))

print("\nNumber of features:", transaction_df.shape[1])

In [ ]:
# Predict fraud probability
fraud_probability = fraud_model.predict_proba(
    transaction_df
)[0][1]

# Use our previously selected threshold
fraud_threshold = 0.90

fraud_prediction = (
    1 if fraud_probability >= fraud_threshold else 0
)

print("Transaction Amount:", sample_transaction["amount"])
print("Sender Previous Balance:", sample_transaction["oldbalanceOrg"])

print("\nFraud Risk Score:", round(fraud_probability, 4))
print("Fraud Risk Percentage:", round(fraud_probability * 100, 2), "%")

print("\nThreshold:", fraud_threshold)

if fraud_prediction == 1:
    print("Decision: FLAG FOR FRAUD INVESTIGATION")
else:
    print("Decision: NOT FLAGGED AT CURRENT THRESHOLD")

In [ ]:
# Step 70: Retrieve policies for the ML prediction

investigation_query = f"""
A TRANSFER transaction has an amount of
{sample_transaction['amount']}.

The sender's balance before the transaction was
{sample_transaction['oldbalanceOrg']}.

Our Random Forest model predicted a fraud risk score
of {fraud_probability:.2%}.

Which fraud investigation policies should we review?
"""

# Convert investigation query into embedding
investigation_embedding = embedding_model.encode(
    investigation_query
).tolist()

# Retrieve relevant policy chunks
policy_results = fraud_collection.query(
    query_embeddings=[investigation_embedding],
    n_results=3,
    include=["documents", "metadatas"]
)

# Combine retrieved policy text
investigation_context = "\n\n".join(
    policy_results["documents"][0]
)

print("ML FRAUD RISK SCORE:", round(fraud_probability * 100, 2), "%")

print("\nRETRIEVED INVESTIGATION POLICIES:")
print(investigation_context)

In [ ]:
# Step 71: Generate fraud investigation report

investigation_prompt = f"""
You are a financial fraud investigation assistant.

Analyze the following synthetic transaction using ONLY
the provided transaction details, ML prediction, and
retrieved investigation policy.

TRANSACTION DETAILS:
Transaction Type: TRANSFER
Transaction Amount: {sample_transaction['amount']}
Sender Balance Before Transaction: {sample_transaction['oldbalanceOrg']}

ML MODEL RESULT:
Fraud Risk Score: {fraud_probability:.2%}
Investigation Threshold: {fraud_threshold:.0%}
Flagged for Investigation: {fraud_prediction == 1}

RETRIEVED FRAUD POLICY:
{investigation_context}

Generate a report with these sections:

1. Transaction Summary
2. ML Fraud Risk Assessment
3. Relevant Investigation Policies
4. Recommended Investigation Actions
5. Final Recommendation

IMPORTANT RULES:
- Do not declare fraud as confirmed.
- Do not invent transaction details or evidence.
- Treat the ML score as a decision-support signal.
- Recommend human investigation when appropriate.
- If information is missing, clearly mention it.
"""

try:
    investigation_response = test_client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=investigation_prompt
    )

    print("AI FRAUD INVESTIGATION REPORT")
    print("=" * 55)
    print(investigation_response.text)

except Exception as e:
    print("Gemini error:", type(e).__name__)
    print(str(e)[:500])

In [ ]:
!pip -q install langgraph

In [ ]:
from typing import TypedDict, Optional

# Shared state for our fraud investigation workflow
class FraudInvestigationState(TypedDict):
    transaction: dict
    fraud_score: Optional[float]
    fraud_prediction: Optional[int]
    retrieved_policy: Optional[str]
    investigation_report: Optional[str]
    validation_status: Optional[str]
    review_required: Optional[bool]

print("LangGraph workflow state created successfully!")
print("State fields:", list(FraudInvestigationState.__annotations__.keys()))

In [ ]:
import pandas as pd

# First LangGraph node: ML fraud scoring
def ml_scoring_node(state: FraudInvestigationState):

    print("ML Scoring Node Started...")

    # Get transaction details from shared state
    transaction = state["transaction"]

    # Convert transaction into model input
    transaction_df = pd.DataFrame([transaction])

    # Maintain the exact feature order used during training
    feature_order = [
        "step",
        "amount",
        "oldbalanceOrg",
        "type_CASH_IN",
        "type_CASH_OUT",
        "type_DEBIT",
        "type_PAYMENT",
        "type_TRANSFER"
    ]

    transaction_df = transaction_df[feature_order]

    # Predict fraud risk using our trained model
    fraud_score = float(
        fraud_model.predict_proba(
            transaction_df.to_numpy()
        )[0][1]
    )

    # Apply our validation-selected threshold
    fraud_prediction = int(fraud_score >= 0.90)

    print("Fraud Score:", round(fraud_score * 100, 2), "%")
    print("Fraud Prediction:", fraud_prediction)

    # Update shared state
    return {
        "fraud_score": fraud_score,
        "fraud_prediction": fraud_prediction
    }

print("ML Scoring Node created successfully!")

In [ ]:
# Second LangGraph node: RAG policy retrieval

def rag_retrieval_node(state: FraudInvestigationState):

    print("RAG Retrieval Node Started...")

    # Read information from shared state
    transaction = state["transaction"]
    fraud_score = state["fraud_score"]

    # Prepare the investigation question
    query = f"""
    A financial transaction has:
    Amount: {transaction['amount']}
    Sender previous balance: {transaction['oldbalanceOrg']}
    Fraud risk score: {fraud_score:.2%}

    Which fraud investigation policies are relevant?
    """

    # Convert question into embedding
    query_embedding = embedding_model.encode(
        query
    ).tolist()

    # Search ChromaDB
    results = fraud_collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["documents", "metadatas"]
    )

    # Combine retrieved policy chunks
    retrieved_policy = "\n\n".join(
        results["documents"][0]
    )

    print("Retrieved policy chunks:",
          len(results["documents"][0]))

    # Update shared state
    return {
        "retrieved_policy": retrieved_policy
    }

print("RAG Retrieval Node created successfully!")

In [ ]:
# Third LangGraph node: Gemini LLM report generation

def llm_generation_node(state: FraudInvestigationState):

    print("Gemini LLM Node Started...")

    # Read data from shared state
    transaction = state["transaction"]
    fraud_score = state["fraud_score"]
    fraud_prediction = state["fraud_prediction"]
    retrieved_policy = state["retrieved_policy"]

    # Prepare prompt for Gemini
    prompt = f"""
    You are a financial fraud investigation assistant.

    Analyze this synthetic transaction using the
    provided transaction details, ML prediction,
    and retrieved investigation policies.

    TRANSACTION:
    Amount: {transaction['amount']}
    Sender Previous Balance: {transaction['oldbalanceOrg']}
    Transaction Type: TRANSFER

    ML RESULT:
    Fraud Risk Score: {fraud_score:.2%}
    Flagged for Investigation: {bool(fraud_prediction)}

    RETRIEVED POLICY:
    {retrieved_policy}

    Generate a report with these sections:

    1. Transaction Summary
    2. ML Fraud Risk Assessment
    3. Relevant Investigation Policies
    4. Recommended Investigation Actions
    5. Final Recommendation

    RULES:
    - Do not declare fraud confirmed.
    - Do not invent evidence or transaction details.
    - Treat the ML score as a decision-support signal.
    - Distinguish suggested checks from verified facts.
    - Do not invent policy requirements.
    - Recommend human review when appropriate.
    """

    # Call Gemini LLM
    response = test_client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    report = response.text

    print("Investigation report generated!")

    # Update shared state
    return {
        "investigation_report": report
    }

print("Gemini LLM Node created successfully!")

In [ ]:
# Fourth LangGraph node: Report Validation

def validation_node(state: FraudInvestigationState):

    print("Validation Node Started...")

    report = state.get("investigation_report") or ""
    fraud_score = state.get("fraud_score")
    fraud_prediction = state.get("fraud_prediction")

    # Basic deterministic validation checks
    required_sections = [
        "Transaction Summary",
        "ML Fraud Risk Assessment",
        "Relevant Investigation Policies",
        "Recommended Investigation Actions",
        "Final Recommendation"
    ]

    missing_sections = [
        section
        for section in required_sections
        if section.lower() not in report.lower()
    ]

    # Check whether the ML prediction exists
    score_valid = (
        fraud_score is not None
        and 0 <= fraud_score <= 1
        and fraud_prediction in (0, 1)
    )

    # Check whether the report has meaningful content
    report_valid = len(report.strip()) >= 100

    if not missing_sections and score_valid and report_valid:
        validation_status = "PASSED"
    else:
        validation_status = "FAILED"

    # High-risk cases or failed validation require review
    review_required = (
        fraud_prediction == 1
        or validation_status == "FAILED"
    )

    print("Validation Status:", validation_status)
    print("Missing Sections:", missing_sections)
    print("Human Review Required:", review_required)

    return {
        "validation_status": validation_status,
        "review_required": review_required
    }

print("Validation Node created successfully!")

In [ ]:
from langgraph.graph import StateGraph, START, END

# Create the workflow using our shared state
workflow = StateGraph(FraudInvestigationState)

# Register our four nodes
workflow.add_node("ml_scoring", ml_scoring_node)
workflow.add_node("rag_retrieval", rag_retrieval_node)
workflow.add_node("llm_generation", llm_generation_node)
workflow.add_node("validation", validation_node)

# Define execution order
workflow.add_edge(START, "ml_scoring")
workflow.add_edge("ml_scoring", "rag_retrieval")
workflow.add_edge("rag_retrieval", "llm_generation")
workflow.add_edge("llm_generation", "validation")
workflow.add_edge("validation", END)

# Compile the graph
fraud_graph = workflow.compile()

print("LangGraph workflow compiled successfully!")
print("Nodes:", list(fraud_graph.nodes.keys()))

In [ ]:
# Step 75: Execute the complete LangGraph workflow

initial_state = {
    "transaction": sample_transaction,
    "fraud_score": None,
    "fraud_prediction": None,
    "retrieved_policy": None,
    "investigation_report": None,
    "validation_status": None,
    "review_required": None
}

try:
    # Execute all connected nodes
    final_state = fraud_graph.invoke(initial_state)

    print("\n" + "=" * 55)
    print("LANGGRAPH EXECUTION COMPLETED")
    print("=" * 55)

    print(
        "Fraud Risk Score:",
        round(final_state["fraud_score"] * 100, 2),
        "%"
    )

    print(
        "Validation Status:",
        final_state["validation_status"]
    )

    print(
        "Human Review Required:",
        final_state["review_required"]
    )

    print("\nAI FRAUD INVESTIGATION REPORT:")
    print(final_state["investigation_report"])

except Exception as e:
    print("Workflow Error:", type(e).__name__)
    print(str(e)[:700])

In [ ]:
# Step 76A: Human review routing node

def human_review_node(state: FraudInvestigationState):

    print("Human Review Node Started...")

    fraud_score = state["fraud_score"]
    validation_status = state["validation_status"]

    print("Fraud Risk:", round(fraud_score * 100, 2), "%")
    print("Validation Status:", validation_status)

    print(
        "Case requires human investigation."
        " No automatic fraud confirmation or transaction hold."
    )

    # This node only records the review requirement.
    # A real human approval process will be added later.
    return {
        "review_required": True
    }

print("Human Review Node created successfully!")

In [ ]:
from langgraph.graph import StateGraph, START, END

# Decide which path to follow after validation
def review_router(state: FraudInvestigationState):

    if state["review_required"]:
        return "human_review"

    return "end"


# Create a new workflow
conditional_workflow = StateGraph(FraudInvestigationState)

# Register all five nodes
conditional_workflow.add_node("ml_scoring", ml_scoring_node)
conditional_workflow.add_node("rag_retrieval", rag_retrieval_node)
conditional_workflow.add_node("llm_generation", llm_generation_node)
conditional_workflow.add_node("validation", validation_node)
conditional_workflow.add_node("human_review", human_review_node)

# Normal execution sequence
conditional_workflow.add_edge(START, "ml_scoring")
conditional_workflow.add_edge("ml_scoring", "rag_retrieval")
conditional_workflow.add_edge("rag_retrieval", "llm_generation")
conditional_workflow.add_edge("llm_generation", "validation")

# Conditional routing
conditional_workflow.add_conditional_edges(
    "validation",
    review_router,
    {
        "human_review": "human_review",
        "end": END
    }
)

# Finish after human review routing
conditional_workflow.add_edge("human_review", END)

# Compile
conditional_fraud_graph = conditional_workflow.compile()

print("Conditional LangGraph workflow compiled successfully!")

In [ ]:
# Step 77: Execute conditional LangGraph workflow

initial_state = {
    "transaction": sample_transaction,
    "fraud_score": None,
    "fraud_prediction": None,
    "retrieved_policy": None,
    "investigation_report": None,
    "validation_status": None,
    "review_required": None
}

try:
    final_state = conditional_fraud_graph.invoke(initial_state)

    print("\n" + "=" * 55)
    print("CONDITIONAL LANGGRAPH EXECUTION COMPLETED")
    print("=" * 55)

    print("Fraud Risk Score:",
          round(final_state["fraud_score"] * 100, 2), "%")

    print("Fraud Prediction:", final_state["fraud_prediction"])

    print("Validation Status:", final_state["validation_status"])

    print("Human Review Required:", final_state["review_required"])

except Exception as e:
    print("Workflow Error:", type(e).__name__)
    print(str(e)[:700])

In [ ]:
# Step 78A: Improved validation with basic safety guardrails

import re

def validation_node(state: FraudInvestigationState):

    print("Enhanced Validation Node Started...")

    report = state.get("investigation_report") or ""
    fraud_score = state.get("fraud_score")
    fraud_prediction = state.get("fraud_prediction")

    required_sections = [
        "Transaction Summary",
        "ML Fraud Risk Assessment",
        "Relevant Investigation Policies",
        "Recommended Investigation Actions",
        "Final Recommendation"
    ]

    missing_sections = [
        section for section in required_sections
        if section.lower() not in report.lower()
    ]

    score_valid = (
        fraud_score is not None
        and 0 <= fraud_score <= 1
        and fraud_prediction in (0, 1)
    )

    report_valid = len(report.strip()) >= 100

    # Detect some potentially unauthorized recommendations
    unsafe_patterns = [
        r"\b(?:should|must|immediately)\s+"
        r"(?:be\s+)?(?:placed\s+on\s+)?(?:a\s+)?"
        r"(?:temporary\s+)?hold\b",
        r"\b(?:automatically|immediately)\s+block\b",
        r"\b(?:automatically|immediately)\s+freeze\b",
        r"\b(?:fraud\s+is\s+confirmed|confirmed\s+fraud)\b"
    ]

    flagged_phrases = [
        pattern for pattern in unsafe_patterns
        if re.search(pattern, report, re.IGNORECASE)
    ]

    if (
        not missing_sections
        and score_valid
        and report_valid
        and not flagged_phrases
    ):
        validation_status = "PASSED"
    else:
        validation_status = "FAILED"

    review_required = (
        fraud_prediction == 1
        or validation_status == "FAILED"
    )

    print("Validation Status:", validation_status)
    print("Missing Sections:", missing_sections)
    print("Safety Flags:", len(flagged_phrases))
    print("Human Review Required:", review_required)

    return {
        "validation_status": validation_status,
        "review_required": review_required
    }

print("Enhanced Validation Node created successfully!")

In [ ]:
# Step 78B: Recompile LangGraph with enhanced validation

conditional_workflow = StateGraph(FraudInvestigationState)

# Register all five nodes
conditional_workflow.add_node("ml_scoring", ml_scoring_node)
conditional_workflow.add_node("rag_retrieval", rag_retrieval_node)
conditional_workflow.add_node("llm_generation", llm_generation_node)
conditional_workflow.add_node("validation", validation_node)
conditional_workflow.add_node("human_review", human_review_node)

# Connect nodes
conditional_workflow.add_edge(START, "ml_scoring")
conditional_workflow.add_edge("ml_scoring", "rag_retrieval")
conditional_workflow.add_edge("rag_retrieval", "llm_generation")
conditional_workflow.add_edge("llm_generation", "validation")

# Conditional routing
conditional_workflow.add_conditional_edges(
    "validation",
    review_router,
    {
        "human_review": "human_review",
        "end": END
    }
)

conditional_workflow.add_edge("human_review", END)

# Compile updated workflow
conditional_fraud_graph = conditional_workflow.compile()

print("Enhanced LangGraph workflow compiled successfully!")


In [ ]:
# Step 79: Test enhanced validation using a sample unsafe report

test_state = {
    "transaction": sample_transaction,
    "fraud_score": 0.995,
    "fraud_prediction": 1,
    "retrieved_policy": "Human investigation is required.",
    "investigation_report": """
    1. Transaction Summary
    A transfer of 90,000 was initiated.

    2. ML Fraud Risk Assessment
    The fraud risk score is 99.5%.

    3. Relevant Investigation Policies
    High-risk transactions require human investigation.

    4. Recommended Investigation Actions
    Investigators should review the transaction.

    5. Final Recommendation
    The transaction should be placed on a temporary hold.
    """,
    "validation_status": None,
    "review_required": None
}

result = validation_node(test_state)

print("\nTEST RESULT")
print("Validation:", result["validation_status"])
print("Human Review:", result["review_required"])

In [ ]:
# Step 80: Positive validation test

safe_test_state = test_state.copy()

safe_test_state["investigation_report"] = """
1. Transaction Summary
A transfer of 90,000 was initiated from an account
with a previous balance of 95,000.

2. ML Fraud Risk Assessment
The ML fraud risk score is 99.5%.
This score does not confirm fraudulent activity.

3. Relevant Investigation Policies
The investigation policy recommends reviewing
transaction amounts, sender balances and risk indicators.

4. Recommended Investigation Actions
A human investigator should examine the transaction
and gather additional evidence.

5. Final Recommendation
Refer this transaction for human investigation.
Do not treat the ML prediction as confirmed fraud.
"""

result = validation_node(safe_test_state)

print("\nPOSITIVE TEST RESULT")
print("Validation:", result["validation_status"])
print("Human Review:", result["review_required"])

In [ ]:
# Step 80A: Fix the confirmed-fraud pattern

unsafe_patterns = [
    r"\b(?:should|must|immediately)\s+"
    r"(?:be\s+)?(?:placed\s+on\s+)?(?:a\s+)?"
    r"(?:temporary\s+)?hold\b",

    r"\b(?:automatically|immediately)\s+block\b",

    r"\b(?:automatically|immediately)\s+freeze\b",

    r"\bfraud\s+is\s+confirmed\b",

    r"(?<!not )(?<!no )\bconfirmed\s+fraud\b"
]

import re

# Test our updated pattern against the safe report
flagged_phrases = [
    pattern for pattern in unsafe_patterns
    if re.search(
        pattern,
        safe_test_state["investigation_report"],
        re.IGNORECASE
    )
]

print("Safety Flags:", len(flagged_phrases))
print("Flagged Patterns:", flagged_phrases)

In [ ]:
import re

def validation_node(state: FraudInvestigationState):

    print("Enhanced Validation Node Started...")

    report = state.get("investigation_report") or ""
    fraud_score = state.get("fraud_score")
    fraud_prediction = state.get("fraud_prediction")

    required_sections = [
        "Transaction Summary",
        "ML Fraud Risk Assessment",
        "Relevant Investigation Policies",
        "Recommended Investigation Actions",
        "Final Recommendation"
    ]

    missing_sections = [
        section for section in required_sections
        if section.lower() not in report.lower()
    ]

    score_valid = (
        fraud_score is not None
        and 0 <= fraud_score <= 1
        and fraud_prediction in (0, 1)
    )

    report_valid = len(report.strip()) >= 100

    # Detect specific potentially unauthorized actions
    unsafe_patterns = [
        r"\bshould\s+be\s+placed\s+on\s+(?:a\s+)?(?:temporary\s+)?hold\b",
        r"\bmust\s+be\s+placed\s+on\s+(?:a\s+)?(?:temporary\s+)?hold\b",
        r"\b(?:automatically|immediately)\s+block\b",
        r"\b(?:automatically|immediately)\s+freeze\b",
        r"\bfraud\s+is\s+confirmed\b",
        r"\bthis\s+is\s+(?:a\s+)?confirmed\s+fraud\b"
    ]

    flagged_phrases = [
        pattern for pattern in unsafe_patterns
        if re.search(pattern, report, re.IGNORECASE)
    ]

    validation_status = (
        "PASSED"
        if (
            not missing_sections
            and score_valid
            and report_valid
            and not flagged_phrases
        )
        else "FAILED"
    )

    review_required = (
        fraud_prediction == 1
        or validation_status == "FAILED"
    )

    print("Validation Status:", validation_status)
    print("Missing Sections:", missing_sections)
    print("Safety Flags:", len(flagged_phrases))
    print("Human Review Required:", review_required)

    return {
        "validation_status": validation_status,
        "review_required": review_required
    }


# Test both reports
print("=== NEGATIVE TEST ===")
negative_result = validation_node(test_state)

print("\n=== POSITIVE TEST ===")
positive_result = validation_node(safe_test_state)

print("\nFINAL TEST RESULTS")
print("Negative test:", negative_result["validation_status"])
print("Positive test:", positive_result["validation_status"])

In [ ]:
# Step 81: Recompile LangGraph with updated validation

conditional_workflow = StateGraph(FraudInvestigationState)

# Register nodes
conditional_workflow.add_node("ml_scoring", ml_scoring_node)
conditional_workflow.add_node("rag_retrieval", rag_retrieval_node)
conditional_workflow.add_node("llm_generation", llm_generation_node)
conditional_workflow.add_node("validation", validation_node)
conditional_workflow.add_node("human_review", human_review_node)

# Define execution flow
conditional_workflow.add_edge(START, "ml_scoring")
conditional_workflow.add_edge("ml_scoring", "rag_retrieval")
conditional_workflow.add_edge("rag_retrieval", "llm_generation")
conditional_workflow.add_edge("llm_generation", "validation")

# Conditional decision
conditional_workflow.add_conditional_edges(
    "validation",
    review_router,
    {
        "human_review": "human_review",
        "end": END
    }
)

conditional_workflow.add_edge("human_review", END)

# Compile updated graph
conditional_fraud_graph = conditional_workflow.compile()

print("Final LangGraph workflow compiled successfully!")

In [ ]:
from pathlib import Path

# Application folder in Google Drive
app_dir = Path(
    "/content/drive/MyDrive/capstone-project-1/src/app"
)

app_dir.mkdir(parents=True, exist_ok=True)

print("Streamlit application folder created successfully!")
print("Location:", app_dir)

In [ ]:
!pip -q install streamlit

In [ ]:
import streamlit as st

print("Streamlit installed successfully!")
print("Version:", st.__version__)

In [ ]:
from pathlib import Path

app_file = Path(
    "/content/drive/MyDrive/capstone-project-1/src/app/app.py"
)

app_file.parent.mkdir(parents=True, exist_ok=True)

# Create the file only if it doesn't already exist
if not app_file.exists():
    app_file.touch()

print("Streamlit app.py created successfully!")
print("File location:", app_file)

In [ ]:

from pathlib import Path

app_file = Path(
    "/content/drive/MyDrive/capstone-project-1/src/app/app.py"
)

app_code = r'''
import os
import re
from pathlib import Path
from typing import TypedDict, Optional

import streamlit as st
import pandas as pd
import joblib
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from langgraph.graph import StateGraph, START, END


# ==========================================
# 1. APPLICATION CONFIGURATION
# ==========================================

st.set_page_config(
    page_title="AI Fraud Intelligence",
    page_icon="🛡️",
    layout="wide"
)

st.title("🛡️ Financial Fraud & Risk Intelligence")
st.caption("ML + RAG + Gemini + LangGraph")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/capstone-project-1"
)

MODEL_PATH = (
    PROJECT_DIR / "artifacts/models/random_forest_v1.joblib"
)

VECTOR_DB_PATH = (
    PROJECT_DIR / "artifacts/vector_db"
)

FRAUD_THRESHOLD = 0.90

FEATURE_ORDER = [
    "step",
    "amount",
    "oldbalanceOrg",
    "type_CASH_IN",
    "type_CASH_OUT",
    "type_DEBIT",
    "type_PAYMENT",
    "type_TRANSFER"
]


# ==========================================
# 2. LOAD SAVED COMPONENTS
# ==========================================

@st.cache_resource
def load_resources():

    model = joblib.load(MODEL_PATH)

    embedding_model = SentenceTransformer(
        "all-MiniLM-L6-v2"
    )

    chroma_client = chromadb.PersistentClient(
        path=str(VECTOR_DB_PATH)
    )

    collection = chroma_client.get_collection(
        name="fraud_investigation_policies"
    )

    return model, embedding_model, collection


try:
    fraud_model, embedding_model, fraud_collection = (
        load_resources()
    )
except Exception as error:
    st.error(f"Unable to load ML/RAG resources: {error}")
    st.stop()


# ==========================================
# 3. GEMINI CLIENT
# ==========================================

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    st.error(
        "GEMINI_API_KEY is missing. "
        "Set it before starting Streamlit."
    )
    st.stop()

gemini_client = genai.Client(api_key=api_key)

GEMINI_MODEL = "gemini-3.1-flash-lite"


# ==========================================
# 4. LANGGRAPH SHARED STATE
# ==========================================

class FraudInvestigationState(TypedDict):
    transaction: dict
    fraud_score: Optional[float]
    fraud_prediction: Optional[int]
    retrieved_policy: Optional[str]
    investigation_report: Optional[str]
    validation_status: Optional[str]
    review_required: Optional[bool]


# ==========================================
# 5. ML SCORING NODE
# ==========================================

def ml_scoring_node(state):

    transaction = state["transaction"]

    transaction_df = pd.DataFrame(
        [transaction]
    )[FEATURE_ORDER]

    fraud_score = float(
        fraud_model.predict_proba(
            transaction_df.to_numpy()
        )[0][1]
    )

    fraud_prediction = int(
        fraud_score >= FRAUD_THRESHOLD
    )

    return {
        "fraud_score": fraud_score,
        "fraud_prediction": fraud_prediction
    }


# ==========================================
# 6. RAG RETRIEVAL NODE
# ==========================================

def rag_retrieval_node(state):

    transaction = state["transaction"]

    query = f"""
    Fraud investigation policies for:
    Transaction amount: {transaction['amount']}
    Sender balance: {transaction['oldbalanceOrg']}
    Fraud risk score: {state['fraud_score']:.2%}
    """

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    results = fraud_collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["documents", "metadatas"]
    )

    documents = results["documents"][0]

    retrieved_policy = "\n\n".join(documents)

    return {
        "retrieved_policy": retrieved_policy
    }


# ==========================================
# 7. GEMINI REPORT GENERATION NODE
# ==========================================

def llm_generation_node(state):

    transaction = state["transaction"]

    transaction_type = next(
        (
            name.replace("type_", "")
            for name in FEATURE_ORDER
            if name.startswith("type_")
            and transaction.get(name) == 1
        ),
        "UNKNOWN"
    )

    prompt = f"""
    You are a financial fraud investigation assistant.

    TRANSACTION:
    Amount: {transaction['amount']}
    Sender Previous Balance:
    {transaction['oldbalanceOrg']}
    Transaction Type: {transaction_type}

    ML RESULT:
    Fraud Risk Score: {state['fraud_score']:.2%}
    Flagged for Investigation:
    {bool(state['fraud_prediction'])}

    RETRIEVED POLICY:
    {state['retrieved_policy']}

    Generate these sections:

    1. Transaction Summary
    2. ML Fraud Risk Assessment
    3. Relevant Investigation Policies
    4. Recommended Investigation Actions
    5. Final Recommendation

    Rules:
    - Do not confirm fraud.
    - Do not invent evidence.
    - Do not recommend automatic holds,
      blocking or freezing accounts.
    - Do not invent policy requirements.
    - Treat the ML score as decision support.
    - Recommend human investigation where appropriate.
    """

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )

    if not response.text:
        raise RuntimeError(
            "Gemini returned an empty investigation report."
        )

    return {
        "investigation_report": response.text
    }


# ==========================================
# 8. REPORT VALIDATION NODE
# ==========================================

def validation_node(state):

    report = state.get(
        "investigation_report"
    ) or ""

    fraud_score = state.get("fraud_score")
    fraud_prediction = state.get("fraud_prediction")

    required_sections = [
        "Transaction Summary",
        "ML Fraud Risk Assessment",
        "Relevant Investigation Policies",
        "Recommended Investigation Actions",
        "Final Recommendation"
    ]

    missing_sections = [
        section
        for section in required_sections
        if section.lower() not in report.lower()
    ]

    score_valid = (
        fraud_score is not None
        and 0 <= fraud_score <= 1
        and fraud_prediction in (0, 1)
    )

    report_valid = len(report.strip()) >= 100

    unsafe_patterns = [
        r"\bshould\s+be\s+placed\s+on\s+(?:a\s+)?(?:temporary\s+)?hold\b",
        r"\bmust\s+be\s+placed\s+on\s+(?:a\s+)?(?:temporary\s+)?hold\b",
        r"\b(?:automatically|immediately)\s+block\b",
        r"\b(?:automatically|immediately)\s+freeze\b",
        r"\bfraud\s+is\s+confirmed\b",
        r"\bthis\s+is\s+(?:a\s+)?confirmed\s+fraud\b"
    ]

    safety_flags = [
        pattern
        for pattern in unsafe_patterns
        if re.search(pattern, report, re.IGNORECASE)
    ]

    validation_status = (
        "PASSED"
        if (
            not missing_sections
            and score_valid
            and report_valid
            and not safety_flags
        )
        else "FAILED"
    )

    review_required = (
        fraud_prediction == 1
        or validation_status == "FAILED"
    )

    return {
        "validation_status": validation_status,
        "review_required": review_required
    }


# ==========================================
# 9. HUMAN REVIEW NODE
# ==========================================

def human_review_node(state):

    # Flags the case for human investigation.
    # Does not perform actual human approval.

    return {
        "review_required": True
    }


# ==========================================
# 10. CONDITIONAL ROUTER
# ==========================================

def review_router(state):

    if state["review_required"]:
        return "human_review"

    return "end"


# ==========================================
# 11. BUILD LANGGRAPH WORKFLOW
# ==========================================

@st.cache_resource
def build_workflow():

    workflow = StateGraph(
        FraudInvestigationState
    )

    workflow.add_node(
        "ml_scoring", ml_scoring_node
    )

    workflow.add_node(
        "rag_retrieval", rag_retrieval_node
    )

    workflow.add_node(
        "llm_generation", llm_generation_node
    )

    workflow.add_node(
        "validation", validation_node
    )

    workflow.add_node(
        "human_review", human_review_node
    )

    workflow.add_edge(START, "ml_scoring")
    workflow.add_edge("ml_scoring", "rag_retrieval")
    workflow.add_edge("rag_retrieval", "llm_generation")
    workflow.add_edge("llm_generation", "validation")

    workflow.add_conditional_edges(
        "validation",
        review_router,
        {
            "human_review": "human_review",
            "end": END
        }
    )

    workflow.add_edge("human_review", END)

    return workflow.compile()


fraud_graph = build_workflow()


# ==========================================
# 12. STREAMLIT USER INTERFACE
# ==========================================

tab1, tab2 = st.tabs([
    "🔍 Fraud Investigation",
    "💬 AI Policy Chatbot"
])


with tab1:

    st.subheader("Transaction Details")

    col1, col2 = st.columns(2)

    with col1:
        amount = st.number_input(
            "Transaction Amount",
            min_value=0.0,
            value=90000.0,
            step=100.0
        )

        sender_balance = st.number_input(
            "Sender Previous Balance",
            min_value=0.0,
            value=95000.0,
            step=100.0
        )

    with col2:
        transaction_type = st.selectbox(
            "Transaction Type",
            [
                "TRANSFER",
                "CASH_OUT",
                "PAYMENT",
                "CASH_IN",
                "DEBIT"
            ]
        )

        transaction_step = st.number_input(
            "Transaction Step",
            min_value=1,
            value=100
        )

    if st.button(
        "Analyze Transaction",
        type="primary"
    ):

        transaction = {
            "step": transaction_step,
            "amount": amount,
            "oldbalanceOrg": sender_balance,
            "type_CASH_IN": int(
                transaction_type == "CASH_IN"
            ),
            "type_CASH_OUT": int(
                transaction_type == "CASH_OUT"
            ),
            "type_DEBIT": int(
                transaction_type == "DEBIT"
            ),
            "type_PAYMENT": int(
                transaction_type == "PAYMENT"
            ),
            "type_TRANSFER": int(
                transaction_type == "TRANSFER"
            )
        }

        initial_state = {
            "transaction": transaction,
            "fraud_score": None,
            "fraud_prediction": None,
            "retrieved_policy": None,
            "investigation_report": None,
            "validation_status": None,
            "review_required": None
        }

        try:
            with st.spinner(
                "Running ML + RAG + Gemini + LangGraph..."
            ):
                result = fraud_graph.invoke(
                    initial_state
                )

            st.session_state["last_result"] = result

        except Exception as error:
            st.error(
                f"Investigation failed: {error}"
            )

    if "last_result" in st.session_state:

        result = st.session_state["last_result"]

        st.divider()
        st.subheader("Fraud Risk Assessment")

        metric1, metric2, metric3 = st.columns(3)

        metric1.metric(
            "Fraud Risk Score",
            f"{result['fraud_score']:.2%}"
        )

        metric2.metric(
            "Validation",
            result["validation_status"]
        )

        metric3.metric(
            "Human Review",
            "Required"
            if result["review_required"]
            else "Not Flagged"
        )

        if result["validation_status"] == "FAILED":
            st.warning(
                "Report failed basic validation. "
                "Human verification is required."
            )

        st.subheader("AI Investigation Report")

        st.markdown(
            result["investigation_report"]
        )

        with st.expander(
            "Retrieved Investigation Policies"
        ):
            st.write(
                result["retrieved_policy"]
            )


# ==========================================
# 13. AI POLICY CHATBOT
# ==========================================

with tab2:

    st.subheader(
        "Ask Questions About Fraud Investigation Policies"
    )

    question = st.text_input(
        "Enter your question",
        placeholder=(
            "What should investigators check "
            "for high-risk transfers?"
        )
    )

    if st.button("Ask Gemini") and question.strip():

        try:
            with st.spinner("Searching policy documents..."):

                question_embedding = (
                    embedding_model.encode(
                        question
                    ).tolist()
                )

                search_results = fraud_collection.query(
                    query_embeddings=[
                        question_embedding
                    ],
                    n_results=3,
                    include=["documents"]
                )

                context = "\n\n".join(
                    search_results["documents"][0]
                )

                chatbot_prompt = f"""
                Answer using only the policy context.

                POLICY CONTEXT:
                {context}

                QUESTION:
                {question}

                If the answer is not in the context,
                say that the policy does not provide
                enough information.

                Do not invent policy requirements.
                """

                response = (
                    gemini_client.models.generate_content(
                        model=GEMINI_MODEL,
                        contents=chatbot_prompt
                    )
                )

                st.markdown(
                    response.text
                    or "No response was generated."
                )

                with st.expander(
                    "Retrieved Policy Context"
                ):
                    st.write(context)

        except Exception as error:
            st.error(
                f"Chatbot request failed: {error}"
            )


st.divider()
st.caption(
    "Educational fraud investigation prototype. "
    "ML predictions and AI reports do not confirm fraud "
    "or authorize financial actions."
)
'''

app_file.write_text(app_code, encoding="utf-8")

print("Complete Streamlit application saved successfully!")
print("File:", app_file)
print("Lines:", len(app_code.splitlines()))


In [ ]:
import os
from google.colab import userdata

# Read API key from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in Colab Secrets")

# Make API key available to Streamlit
os.environ["GEMINI_API_KEY"] = api_key

print("Gemini API key configured successfully!")
print("API key is available to the Streamlit process.")

In [ ]:
import subprocess
import os
import time

app_path = "/content/drive/MyDrive/capstone-project-1/src/app/app.py"

# Start Streamlit in the background
streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        app_path,
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true"
    ],
    env=os.environ.copy(),
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Streamlit process started!")
print("Process ID:", streamlit_process.pid)
print("Port: 8501")

In [ ]:
import subprocess
import os
import time

# Stop the previous Streamlit server on port 8501
subprocess.run(
    ["pkill", "-f", "streamlit run"],
    check=False
)

time.sleep(2)

app_path = "/content/drive/MyDrive/capstone-project-1/src/app/app.py"

# Restart Streamlit with Colab-compatible settings
log_handle = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        app_path,
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ],
    env=os.environ.copy(),
    stdout=log_handle,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Streamlit restarted successfully!")
print("Process ID:", streamlit_process.pid)
print("Running on port 8501")
print("Process still running:", streamlit_process.poll() is None)

In [ ]:
from google.colab import output

# Generate the browser-accessible URL
dashboard_url = output.eval_js(
    "google.colab.kernel.proxyPort(8501)"
)

print("Your Streamlit Dashboard URL:")
print(dashboard_url)

In [ ]:
# Step 1: Create a deliberately unsafe report

unsafe_report = """
1. Transaction Summary
Transaction amount: 90000

2. ML Fraud Risk Assessment
Fraud risk score: 99.5%

3. Relevant Investigation Policies
Review the transaction details.

4. Recommended Investigation Actions
All transactions should be placed on a temporary hold.

5. Final Assessment
Human investigation is recommended.
"""

# Use the validation node from our existing LangGraph project
test_state = {
    "fraud_score": 0.995,
    "investigation_report": unsafe_report,
    "review_required": False
}

validation_result = validation_node(test_state)

print("Validation Result:", validation_result)

In [ ]:
# Test LangGraph routing after validation failure

routing_state = {
    "validation_status": "FAILED",
    "review_required": True,
    "fraud_score": 0.995
}

# Use the routing function from our existing graph
next_node = review_router(routing_state)

print("Validation Status:", routing_state["validation_status"])
print("Human Review Required:", routing_state["review_required"])
print("Next Route:", next_node)

In [ ]:
# Test Human Review Node

human_review_state = {
    "fraud_score": 0.995,
    "validation_status": "FAILED",
    "review_required": True
}

result = human_review_node(human_review_state)

print("Human Review Node Result:", result)

In [ ]:
import time
from google import genai
from google.genai import types
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

test_client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=60000)
)

try:
    print("Testing Gemini API...")

    response = test_client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents="Reply with only: Gemini API working"
    )

    print("Gemini Response:", response.text)

except Exception as e:
    print("Gemini Test Failed")
    print("Error Type:", type(e).__name__)
    print("Error:", str(e)[:500])

In [ ]:
import time

print("Starting complete LangGraph workflow...")

start_time = time.time()

try:
    result = fraud_graph.invoke(initial_state)

    print("\n--- FINAL RESULT ---")
    print("Fraud Score:", result.get("fraud_score"))
    print("Validation:", result.get("validation_status"))
    print("Human Review:", result.get("review_required"))
    print("Report Generated:", bool(result.get("investigation_report")))
    print("Execution Time:", round(time.time() - start_time, 2), "seconds")

except Exception as e:
    print("\nWorkflow failed")
    print("Error Type:", type(e).__name__)
    print("Error:", str(e)[:500])
    print("Execution Time:", round(time.time() - start_time, 2), "seconds")

In [ ]:
import os

project_path = "/content/drive/MyDrive/capstone-project-1"

for root, dirs, files in os.walk(project_path):
    for file in files:
        if (
            "audit" in file.lower()
            or "audit" in root.lower()
        ):
            print(os.path.join(root, file))